# Quantum-limited imaging using diffractive optical neural networks — Fig. 4

Companion notebook for

> A. Warke, A. Zhang, A. I. Lvovsky, *Quantum-limited imaging using diffractive optical neural networks*, [arXiv:2608.12300](https://arxiv.org/abs/2608.12300).

**Fig. 4 — 2-D imaging at the quantum limit.** Three objects on the same $L = 2\,\mu$m field of view, all at $N = 64$: a **cell-like** synthetic pattern (three cosine modes, $4$ photons/px), a defected **atom array** (200 nm pitch, $10^4$ photons/px), and a **diatom** built from an SEM image (`data/SEM_orig.npy`, $10^4$ photons/px). Per row the figure shows: **(i)** ground truth, **(ii)** the per-mode CRB ratio $[F^{-1}_{\rm DONN}]_{mm}/[F^{-1}_{\rm DI}]_{mm}$ (per-photon, budget-independent), **(iii)** the direct-imaging reconstruction, **(iv)** the DONN reconstruction, and **(v)** the precision-bound bars.

Both reconstructions use the **same locally-unbiased linear (score) estimator** on Poisson counts — the only difference between (iii) and (iv) is the detection channel (raw image-plane intensities vs the trained DONN outputs). Raw flat-fielded camera frames are produced separately in the last cell; they are not part of the main figure.

**Two pipelines.**
- *Cell-like row:* its own 3-mode problem (`select_modes`) with a coarser Gram cut (`SYN_OPTICS_TOL = 1e-2`) so the Nagaoka–Hayashi SDP block $(M{+}1)K \approx 800$ stays solvable — this row carries the **NHCRB** bar (`nhcrb=True` for `syn2` only).
- *Device (atom array + diatom):* one full-band mode set (`select_all_modes`, `DEVICE_OPTICS_TOL = 1e-4`) and **one shared DONN, trained on the `blank` object** — the flat background $a_0$ itself. $\rho_0$ and all Fisher matrices are object-independent, so masks, calibrations and bounds are computed once and applied unchanged to both rows (their CRB-ratio panels are identical by construction).

**Precision split:** the DONN mask search runs in `float32`/`complex64`; bounds, calibrations, estimators and Monte Carlo are NumPy float64.

**Inputs:** `data/SEM_orig.npy` (SEM image for the diatom row).
**Outputs:** `figures/WZL_Fig4.png` / `.svg`; `data/WZL_Fig4_syn.npz` and `data/WZL_Fig4_objects.npz` (small, commit these — the figure cell reads only these two), `data/WZL_Fig4_device.npz` (heavy local cache with the trained device: >100 MB at full size, keep it out of the repository), and `data/WZL_Fig4_dicam.npz` from the optional camera cell.

**Requirements:** `numpy`, `scipy`, `torch`, `matplotlib`, `cvxpy` (Clarabel), `pillow`. A CUDA GPU is used automatically if available. The two cells marked **[slow]** each train one `NPLANES`-plane 2-D DONN (Adam + L-BFGS polish, `RESTARTS` restarts); the syn cell additionally solves the NHCRB SDP (memory-heavy). The objects cell only reuses saved masks, and the **[fast]** figure cell reads only the two small `.npz` files — run everything except the **[slow]** cells to re-plot from saved data.

## Configuration

In [ ]:
%matplotlib inline
import time
import numpy as np
import torch, torch.nn as nn
import cvxpy as cp
from scipy.fft import dctn, idctn
from scipy.special import j1
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.patches import Rectangle
from matplotlib.ticker import FormatStrFormatter, FuncFormatter
from PIL import Image
from pathlib import Path

# device / precision
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RDTYPE = torch.float32        # real dtype    -> torch.float32 for a large GPU speed-up
CDTYPE = torch.complex64      # complex dtype -> torch.complex64 (must match RDTYPE)
BATCHED_FISHER = True         # vectorise the M-loop in fisher(): faster on GPU, more memory

torch.set_default_dtype(RDTYPE)
if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = False   # keep full precision if RDTYPE=float32
    print(f"training on {torch.cuda.get_device_name(0)}  ({RDTYPE})", flush=True)
else:
    print("CUDA not available - training on CPU", flush=True)

# optics / budget
N, lam, NA, a0 = 64, 0.54, 1.4, 1.0
NPH_ppx_obj1   = 4            # cell-like
NPH_ppx_obj2   = 10000        # atom array
NPH_ppx_obj3   = 10000        # diatom
MC_RUNS        = 5000

DEVICE_OPTICS_TOL = 1e-4      # Gram cut for the shared full-band device (rows 2-3)
SYN_OPTICS_TOL    = 1e-2      # coarser cut for the cell-like row, so the NHCRB SDP
                              # block (M+1)*K stays ~800 (as in the published run)

# DONN training
NPLANES  = 128     # P
EPOCHS   = 3000    # Adam steps per restart; changed to 3000 from 1500 after arXiv v1 upload because it may have been still converging...
RESTARTS = 3       # random restarts, best kept
LR0      = 0.05    # cosine-annealed to 0
POLISH   = True    # L-BFGS (strong Wolfe) on the best Adam solution
LBFGS_IT = 400

# NHCRB solver
SDP_EPS   = 1e-4     # syn (block ~800) converges here; tighter needs ~7000 more iters
SDP_ITERS = 20000

# output paths
Path("data").mkdir(exist_ok=True); Path("figures").mkdir(exist_ok=True)
SYN_NPZ     = "data/WZL_Fig4_syn.npz"       # cell-like row (3-mode pipeline)
DEVICE_NPZ  = "data/WZL_Fig4_device.npz"    # shared full-band device (rows 2-3)
OBJECTS_NPZ = "data/WZL_Fig4_objects.npz"   # atoms + diatom results
DICAM_NPZ   = "data/WZL_Fig4_dicam.npz"     # optional camera-frame reference

## Objects

`OBJECTS` selects the rows (order = row order); the `blank` entry is the flat background $a_0$ itself — it is not a figure row (`nph=None`) but the object the shared device DONN is trained on. Objects are sums of 2-D cosines (`MODES`), or custom intensity maps in `CUSTOM`: the defected atom array, and the diatom built by `photo_object` from `data/SEM_orig.npy` — centre-cropped, area-averaged onto the $N \times N$ grid, and rescaled to a peak modulation of `PHOTO_CONTRAST` about the flat background $a_0$ (mean pinned to $a_0$, strictly positive).

In [ ]:
OBJECTS = {
    "syn2":   dict(L=2.0, nmodes=3,    label="cell-like",  show_err=True,  nhcrb=True,  nph=NPH_ppx_obj1),
    # "blank" is the flat background a0 itself: the shared device DONN is trained on it (object-independent by construction) and then applied unchanged to the atom array and the diatom.
    "blank":  dict(L=2.0, nmodes=None, label="blank (trains the device)", show_err=False, nhcrb=False, nph=None), 
    "atoms":  dict(L=2.0, nmodes=None, label="atom array", show_err=False, nhcrb=False, nph=NPH_ppx_obj2),
    "diatom": dict(L=2.0, nmodes=None, label="diatom",     show_err=False, nhcrb=False, nph=NPH_ppx_obj3),
}
ROW_OBJECTS = [n for n, s in OBJECTS.items() if s["nph"] is not None]   # figure rows

# (m_x, m_y) -> amplitude, on top of the flat background a0; synthetically constructed such that NHCRB remains computable
MODES = {"syn2": [((14, 14), 0.2), ((19, 5), 0.2), ((5, 19), 0.2)]}

def _blank(L, X, Y):
    return np.full((N, N), a0, dtype=float)

def _atoms(L, X, Y, pitch=0.2, n_side=10, sigma=0.045, amp=0.25, remove_atoms=[(0,7),(4,2),(9,5),(1,8),(6,0),(3,4),(8,1),(2,9),(5,6),(7,3),(0,1),(4,8),(9,0),(1,5),(6,7),(3,2),(8,6),(2,4),(5,9),(7,0),(0,6),(4,1),(9,7),(1,2),(6,5)]):
    # 200 nm pitch, defected atom array
    o  = np.full((N, N), a0, dtype=float)
    remove_atoms = set(remove_atoms) if remove_atoms is not None else set()
    c0 = L / 2 - (n_side - 1) / 2 * pitch
    for i in range(n_side):
        for j in range(n_side):
            if (i, j) in remove_atoms:
                continue
            xc, yc = c0 + i * pitch, c0 + j * pitch
            o += amp * np.exp(-((X - xc)**2 + (Y - yc)**2) / (2 * sigma**2))
    return o

# ------------------------------------------------------------------ SEM photo object
PHOTO_PATH     = "data/SEM_orig.npy"  # .npy or .jpg/.png; RGB or greyscale, any size
PHOTO_CONTRAST = 0.5               # peak modulation about the a0 background
PHOTO_SPAN     = None              # None = fill the FOV; else the photo's width in um
PHOTO_CENTRE   = None              # None = centred; else (x, y) in um
PHOTO_INVERT   = False
PHOTO_FLIP_V   = True              # image row 0 is the top; we plot origin="lower"

def _load_field(path):
    if str(path).lower().endswith(".npy"):
        a = np.load(path).astype(float)
    else:
        a = np.asarray(Image.open(path).convert("L"), dtype=float)
    if a.ndim == 3:                                     # RGB/RGBA -> luminance
        a = a[..., :3] @ np.array([0.299, 0.587, 0.114])
    return a

def photo_object(path, L, span=None, centre=None, contrast=0.5,
                 invert=False, flip_v=True, filt=Image.BOX):
    span   = L if span is None else span
    centre = (L/2, L/2) if centre is None else centre
    a = _load_field(path)
    h, w = a.shape; s = min(h, w)                       # centre-crop to square
    a = a[(h-s)//2:(h-s)//2+s, (w-s)//2:(w-s)//2+s]
    n = max(1, int(round(span/L*N)))
    if n != s:                                          # BOX = area average
        a = np.asarray(Image.fromarray(a.astype(np.float32)).resize((n, n), filt), dtype=float)
    if flip_v: a = a[::-1]
    if invert: a = a.max() - a
    d  = a - a.mean(); d *= contrast*a0/max(np.abs(d).max(), 1e-12)
    o  = np.full((N, N), a0, dtype=float)
    j0 = int(round((centre[0]-span/2)/L*N)); i0 = int(round((centre[1]-span/2)/L*N))
    js, is_ = slice(max(j0, 0), min(j0+n, N)), slice(max(i0, 0), min(i0+n, N))
    o[is_, js] = a0 + d[is_.start-i0:is_.stop-i0, js.start-j0:js.stop-j0]
    return o

def _diatom(L, X, Y):
    return photo_object(PHOTO_PATH, L, span=PHOTO_SPAN, centre=PHOTO_CENTRE,
                        contrast=PHOTO_CONTRAST, invert=PHOTO_INVERT, flip_v=PHOTO_FLIP_V)

CUSTOM = {"blank": _blank, "atoms": _atoms, "diatom": _diatom}   # objects that are not sums of cosines

def make_object(name):
    L  = OBJECTS[name]["L"]
    x  = (np.arange(N) + 0.5) * (L / N)
    X, Y = np.meshgrid(x, x, indexing="xy")
    if name in CUSTOM:
        o = np.asarray(CUSTOM[name](L, X, Y), dtype=float)
        assert o.shape == (N, N) and o.min() > 0, "object must be positive (it is an intensity)"
    else:
        o = np.full((N, N), a0)
        for (mx, my), am in MODES[name]:
            o += am * np.cos(np.pi * mx / L * X) * np.cos(np.pi * my / L * Y)
    return o / L**2

def cosine_mode(m, L, amp=1.0):
    x = (np.arange(N) + 0.5) * (L / N)
    X, Y = np.meshgrid(x, x, indexing="xy")
    return amp * np.cos(np.pi * m[0] / L * X) * np.cos(np.pi * m[1] / L * Y) / L**2

dct_alpha = lambda k: np.sqrt(1.0 / N) if k == 0 else np.sqrt(2.0 / N)
dct_to_amp = lambda C, ux, uy, L: (L**2) * dct_alpha(ux) * dct_alpha(uy) * C[uy, ux]
amp_to_dct = lambda a, ux, uy, L: a / ((L**2) * dct_alpha(ux) * dct_alpha(uy))

## 2D optics and mode selection for Fig. 4a

In [ ]:
def jinc(u):
    return np.where(u < 1e-9, 1.0, 2*j1(np.maximum(u,1e-12))/np.maximum(u,1e-12))

_OPTICS_CACHE = {}
def optics_exact_2d(L, det_span=2, tol=DEVICE_OPTICS_TOL, chunk=4096):
    key = (L, det_span, tol)
    if key in _OPTICS_CACHE: return _OPTICS_CACHE[key]
    dxx = L/N; nc = NA/lam
    x = (np.arange(N)+0.5)*dxx
    X,Y = np.meshgrid(x,x,indexing="xy"); px,py = X.ravel(), Y.ravel()
    Nd = det_span*N
    xd = (np.arange(Nd)+0.5)*dxx - (Nd-N)//2*dxx
    XD,YD = np.meshgrid(xd,xd,indexing="xy"); qx,qy = XD.ravel(), YD.ravel()
    psi = np.empty((N*N, Nd*Nd))
    for s in range(0, Nd*Nd, chunk):
        e = min(s+chunk, Nd*Nd)
        R = np.sqrt((qx[None,s:e]-px[:,None])**2 + (qy[None,s:e]-py[:,None])**2)
        psi[:, s:e] = np.sqrt(np.pi)*nc*jinc(2*np.pi*nc*R)*dxx
    Gd = psi @ psi.T
    ev, V = np.linalg.eigh(Gd); keep = ev > tol*ev.max()
    evk, Vk = ev[keep], V[:,keep]
    A_states = Vk*np.sqrt(evk)                    # (N^2, K)
    B        = (Vk/np.sqrt(evk)).T @ psi          # (K, Nd^2)
    _OPTICS_CACHE[key] = (A_states, B, int(keep.sum()), psi, Nd)
    return _OPTICS_CACHE[key]

def object_operator(O_spatial, A_states):
    return A_states.T @ (O_spatial.ravel()[:, None] * A_states)

def select_modes(obj, nmodes, L, trn, A_states, indep_tol=0.35):
    C = np.abs(dctn(obj, type=2, norm="ortho"))
    mi = np.arange(N)
    Fx, Fy = np.meshgrid(mi / (2 * L), mi / (2 * L))
    Cr = C.copy(); Cr[0, 0] = 0.0
    Cr[np.sqrt(Fx**2 + Fy**2) > 2 * NA / lam] = 0.0
    rank  = np.argsort(Cr.ravel())[::-1][:6 * nmodes]
    cands = [(int(i % N), int(i // N)) for i in rank]
    kept, drs, basis = [], [], []
    for m in cands:
        dr  = object_operator(cosine_mode(m, L), A_states) / trn
        nrm = np.linalg.norm(dr)
        if nrm < 1e-12:                       # dead on the discrete pupil
            continue
        g = dr.ravel() / nrm
        r = g - sum(np.dot(b, g) * b for b in basis) if basis else g
        if np.linalg.norm(r) < indep_tol:    # not identifiable alongside the kept set
            continue
        kept.append(m); drs.append(dr)
        basis.append(np.asarray(r) / np.linalg.norm(r))
        if len(kept) == nmodes:
            break
    return kept, drs

def select_all_modes(L, NA, lam, N, trn, A_states, indep_tol=0.00035, max_modes=None, force=()):
    mi = np.arange(N)
    Fx, Fy = np.meshgrid(mi / (2 * L), mi / (2 * L))
    inside = np.sqrt(Fx**2 + Fy**2) <= 2 * NA / lam
    inside[0, 0] = False

    uy_idx, ux_idx = np.where(inside)
    cands = [(int(a), int(b)) for a, b in zip(ux_idx, uy_idx)]
    cands.sort(key=lambda m: m[0]**2 + m[1]**2)
    force = [tuple(m) for m in force]
    cands = force + [m for m in cands if m not in force]

    kept, drs, basis = [], [], []
    for m in cands:
        dr  = object_operator(cosine_mode(m, L), A_states) / trn
        nrm = np.linalg.norm(dr)
        if nrm < 1e-12:
            continue
        g = dr.ravel() / nrm
        r = g - sum(np.dot(b, g) * b for b in basis) if basis else g
        if np.linalg.norm(r) < indep_tol:
            continue
        kept.append(m); drs.append(dr)
        basis.append(np.asarray(r) / np.linalg.norm(r))
        if max_modes is not None and len(kept) == max_modes:
            break
    return kept, drs

### Diagnostic: ground truth vs band limit vs achievable set — *optional*

In [ ]:
def plot_gt_bandlimited(name, dct_vmin=None, indep_tol=0.35):
    L = OBJECTS[name]["L"]
    obj = make_object(name)
    C_gt = dctn(obj, type=2, norm="ortho")
    C_gt_abs = np.abs(C_gt)

    f_c = 2 * NA / lam
    m_cut = f_c * 2 * L
    m_coh = f_c * L
    m_idx = np.arange(N)
    Fx, Fy = np.meshgrid(m_idx / (2 * L), m_idx / (2 * L))
    optical_mask = np.sqrt(Fx**2 + Fy**2) <= f_c
    C_cut = C_gt * optical_mask
    num_kept = int(optical_mask.sum())
    obj_cutoff = idctn(C_cut, type=2, norm="ortho")

    tol = SYN_OPTICS_TOL if OBJECTS[name]["nmodes"] else DEVICE_OPTICS_TOL
    A_states, B, K, psi, Nd = optics_exact_2d(L, tol=tol)
    bg  = np.full((N, N), a0) / L**2
    trn = np.trace(object_operator(bg, A_states))

    if OBJECTS[name]["nmodes"]:
        kept, _ = select_modes(obj, OBJECTS[name]["nmodes"], L, trn, A_states,
                               indep_tol=indep_tol)
    else:
        kept, _ = select_all_modes(L, NA, lam, N, trn, A_states)

    # achievable object: selected modes + calibrated DC background
    C_sel = np.zeros_like(C_gt)
    C_sel[0, 0] = C_gt[0, 0]                      # known/calibrated background
    for ux, uy in kept:
        C_sel[uy, ux] = C_gt[uy, ux]
    obj_sel = idctn(C_sel, type=2, norm="ortho")

    img_vmin, img_vmax = obj.min(), obj.max()
    dct_vmax = C_gt_abs.max()
    dct_vmin = dct_vmin or dct_vmax * 1e-4

    theta = np.linspace(0, np.pi / 2, 400)
    xc, yc = m_cut * np.cos(theta), m_cut * np.sin(theta)
    xc_coh, yc_coh = m_coh * np.cos(theta), m_coh * np.sin(theta)

    fig, axes = plt.subplots(2, 3, figsize=(19, 12))

    # row 0: images
    axes[0, 0].imshow(obj, cmap="inferno", extent=[0, L, 0, L], origin="lower",
                       vmin=img_vmin, vmax=img_vmax)
    axes[0, 0].set_title("Ground Truth Image", fontsize=18)
    axes[0, 1].imshow(obj_cutoff, cmap="inferno", extent=[0, L, 0, L], origin="lower",
                       vmin=img_vmin, vmax=img_vmax)
    axes[0, 1].set_title(f"Bandlimited ({num_kept} modes)", fontsize=18)
    axes[0, 2].imshow(obj_sel, cmap="inferno", extent=[0, L, 0, L], origin="lower",
                       vmin=img_vmin, vmax=img_vmax)
    axes[0, 2].set_title(f"Max. achievable resolution ({len(kept)} modes)", fontsize=18)

    for ax in axes[0, :]:
        ax.set_xlabel("x (µm)"); ax.set_ylabel("y (µm)")
        ax.set_xlim(0, L); ax.set_ylim(0, L)

    # row 1: DCT maps
    im_gt = axes[1, 0].imshow(np.maximum(C_gt_abs, 1e-18), origin="lower", cmap="viridis",
                               norm=LogNorm(vmin=dct_vmin, vmax=dct_vmax),
                               extent=[-0.5, N - 0.5, -0.5, N - 0.5])
    axes[1, 0].set_title("Ground Truth DCT-II Map", fontsize=18)
    fig.colorbar(im_gt, ax=axes[1, 0], fraction=0.046, pad=0.04)

    im_cut = axes[1, 1].imshow(np.maximum(np.abs(C_cut), 1e-18), origin="lower", cmap="viridis",
                                norm=LogNorm(vmin=dct_vmin, vmax=dct_vmax),
                                extent=[-0.5, N - 0.5, -0.5, N - 0.5])
    axes[1, 1].set_title(f"Bandlimited DCT ({num_kept} modes)", fontsize=18)
    fig.colorbar(im_cut, ax=axes[1, 1], fraction=0.046, pad=0.04)

    im_sel = axes[1, 2].imshow(np.maximum(np.abs(C_cut), 1e-18), origin="lower", cmap="viridis",
                                norm=LogNorm(vmin=dct_vmin, vmax=dct_vmax),
                                extent=[-0.5, N - 0.5, -0.5, N - 0.5])
    axes[1, 2].set_title(f"Selected modes ({len(kept)})", fontsize=18)
    fig.colorbar(im_sel, ax=axes[1, 2], fraction=0.046, pad=0.04)
    for ux, uy in kept:
        axes[1, 2].add_patch(Rectangle((ux - .5, uy - .5), 1, 1, fill=False, ec="r", lw=1.8))

    for ax in axes[1, :]:
        ax.plot(xc, yc, "w-", lw=1.5, label="Incoherent Cutoff")
        ax.plot(xc_coh, yc_coh, "w--", lw=1.5, label="Coherent Cutoff")
        ax.set_xlabel("mode index m"); ax.set_ylabel("mode index n")
        ax.set_xlim(-0.5, m_cut + 4.5); ax.set_ylim(-0.5, m_cut + 4.5)
        ax.legend(loc="upper right", fontsize=13)

    fig.suptitle(f"{name}  (L={L})", fontsize=18)
    plt.tight_layout()
    plt.show()

plot_gt_bandlimited("atoms")

## 2-dimensional DONN

`MPLC2D` — a cascade of `NPLANES` trainable phase masks separated by optical 2-D Fourier transforms, acting on the $K$ pupil modes; the loss is the joint $\mathrm{Tr}\,F^{-1}$. `train_donn` runs `RESTARTS` restarts of cosine-annealed Adam followed by an L-BFGS (strong-Wolfe) polish and keeps the best.

In [ ]:
class MPLC2D(nn.Module):
    def __init__(self, n_planes, Nd, init=None, gen=None, device=DEVICE):
        super().__init__(); self.n = n_planes
        if init is None:
            init = [torch.randn(Nd, Nd, generator=gen, dtype=RDTYPE)*0.5 for _ in range(n_planes)]
        self.masks = nn.ParameterList([
            nn.Parameter(p.detach().clone().to(device=device, dtype=RDTYPE)) for p in init])

    def V(self, B):
        psi = B                                          # (K, N, N) complex, on `device`
        for p in range(self.n):
            psi = psi * torch.exp(1j * self.masks[p].to(CDTYPE))
            psi = torch.fft.fftshift(
                    torch.fft.fft2(torch.fft.ifftshift(psi, dim=(-2, -1)), norm="ortho"),
                    dim=(-2, -1))
        return psi.reshape(psi.shape[0], -1)             # (K, D)

    def fisher(self, B, rho0, drho):
        V  = self.V(B); cV = V.conj()
        u  = torch.clamp(torch.real((cV * (rho0 @ V)).sum(0)), min=1e-12)
        if BATCHED_FISHER:
            # one batched matmul instead of M small ones - far fewer kernel launches
            du = torch.real((cV.unsqueeze(0) * torch.matmul(drho, V)).sum(1))  # (M, D)
        else:
            du = torch.stack([torch.real((cV * (drho[m] @ V)).sum(0))
                              for m in range(drho.shape[0])])
        return (du / u) @ du.T

    def loss(self, B, rho0, drho):
        F = self.fisher(B, rho0, drho)
        eye = torch.eye(F.shape[0], dtype=F.dtype, device=F.device)
        return torch.trace(torch.linalg.inv(F + 1e-12 * eye))


def train_donn(Bt, Nd, rho0, drho, n_planes=NPLANES, epochs=EPOCHS,
               restarts=RESTARTS, lr=LR0, polish=POLISH, seed0=0, verbose=True,
               device=DEVICE):
    K = rho0.shape[0]
    B     = torch.tensor(Bt.reshape(K, Nd, Nd), dtype=CDTYPE, device=device)
    rho_t = torch.tensor(rho0,                  dtype=CDTYPE, device=device)
    dr_t  = torch.tensor(np.stack(drho),        dtype=CDTYPE, device=device)

    def _sync():
        if device.type == "cuda":
            torch.cuda.synchronize()

    best_val, best_masks = np.inf, None
    for r in range(restarts):
        _sync(); t0 = time.time()
        gen = torch.Generator().manual_seed(seed0 + r)   # CPU generator, masks moved to device
        model = MPLC2D(n_planes, Nd, gen=gen, device=device)
        opt   = torch.optim.Adam(model.parameters(), lr=lr)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
        for ep in range(epochs):
            opt.zero_grad()
            v = model.loss(B, rho_t, dr_t)
            v.backward(); opt.step(); sched.step()
            if verbose and ep % 100 == 0:
                print(f"    restart {r}  ep {ep:5d}  Tr[F^-1] = {v.item():.2f}", flush=True)
        adam_val = float(model.loss(B, rho_t, dr_t).detach())

        if polish:
            lb = torch.optim.LBFGS(model.parameters(), max_iter=LBFGS_IT,
                                   line_search_fn="strong_wolfe")
            def closure():
                lb.zero_grad()
                v = model.loss(B, rho_t, dr_t)
                v.backward()
                return v
            lb.step(closure)

        val = float(model.loss(B, rho_t, dr_t).detach())
        _sync()
        if verbose:
            print(f"  restart {r}: Adam {adam_val:.2f} -> polished {val:.2f}"
                  f"  ({time.time()-t0:.0f}s)", flush=True)
        if val < best_val:
            best_val  = val
            best_masks = [p.detach().clone() for p in model.masks]

    model = MPLC2D(n_planes, Nd, init=best_masks, device=device)
    V = model.V(B).detach().cpu().numpy()      # back to NumPy for the downstream cells
    if verbose:
        print(f"  best over {restarts} restarts: {best_val:.2f}", flush=True)
    if device.type == "cuda":
        del B, rho_t, dr_t
        torch.cuda.empty_cache()
    return best_val, V, best_masks

## Bounds, calibrations, direct imaging

`qcrb_scalar` (SLD QFI on the truncated support), `nhcrb_sdp` (Nagaoka–Hayashi SDP, Clarabel; block side $(M{+}1)K$), `di_machinery` (image-plane Poisson channel: $u_0$, derivatives, Fisher matrix), `donn_calibration` (same triple for the DONN channel). Everything downstream treats the two channels identically.

In [ ]:
def truncate_support(rho, drl, tol=1e-6):
    ev, U = np.linalg.eigh(rho)
    Uk = U[:, ev > tol*ev.max()]
    return Uk.T @ rho @ Uk, [Uk.T @ d @ Uk for d in drl], Uk.shape[1]

def qcrb_scalar(rho, drl, tol=1e-6):
    rho, drl, K = truncate_support(rho, drl, tol)
    ev, U = np.linalg.eigh(rho)
    inv = 1.0/(ev[:,None]+ev[None,:])
    dU = [U.T @ d @ U for d in drl]
    Q = np.array([[2*np.sum(dU[a]*dU[b]*inv) for b in range(len(drl))]
                  for a in range(len(drl))])
    return np.trace(np.linalg.inv(Q))

def nhcrb_sdp(rho, drl, trunc_tol=1e-4, solver=cp.CLARABEL, eps=1e-4,
              max_iters=20000, verbose=False):
    rho, drl, K = truncate_support(rho, drl, trunc_tol)
    M = len(drl)
    dn = np.array([np.linalg.norm(d) for d in drl])
    drt = [d/n for d,n in zip(drl,dn)]; w = 1.0/dn**2
    Xs = [cp.Variable((K,K), symmetric=True) for _ in range(M)]
    Ls = {(j,k): cp.Variable((K,K), symmetric=True) for j in range(M) for k in range(j,M)}
    gL = lambda j,k: Ls[(j,k)] if j<=k else Ls[(k,j)].T
    cons = [cp.trace(drt[j]@Xs[k]) == (1.0 if j==k else 0.0) for j in range(M) for k in range(M)]
    cons.append(cp.bmat([[gL(j,k) for k in range(M)]+[Xs[j]] for j in range(M)]
                        + [[Xs[m].T for m in range(M)]+[np.eye(K)]]) >> 0)
    prob = cp.Problem(cp.Minimize(sum(w[j]*cp.trace(rho@Ls[(j,j)]) for j in range(M))), cons)
    kw = {} if solver==cp.CLARABEL else dict(eps=eps, max_iters=max_iters)
    prob.solve(solver=solver, verbose=verbose, **kw)
    return float(prob.value), K

def di_machinery(modes, L, psi):
    P  = psi**2                                    # (N^2 emitters, Nd^2 detector pixels)
    bg = (np.full((N, N), a0)/L**2).ravel()
    ub = bg @ P
    nb = ub.sum()
    u0 = ub/nb
    dM = np.stack([(cosine_mode(m, L).ravel() @ P)/nb for m in modes])
    return u0, dM, (dM/u0) @ dM.T, nb

def donn_calibration(V, rho0, drho):
    cV = V.conj()
    u0 = np.clip(np.real(np.einsum("ip,ij,jp->p", V, rho0, cV)), 1e-15, None)
    dM = np.array([np.real(np.einsum("ip,ij,jp->p", V, d, cV)) for d in drho])
    F  = np.array([[np.sum(dM[i] * dM[j] / u0) for j in range(len(drho))]
                   for i in range(len(drho))])
    return u0, dM, F

### Cell-like row — **[slow]**

Own 3-mode pipeline at `SYN_OPTICS_TOL`: mode selection, DONN training, QCRB + **NHCRB SDP**, both matched-estimator reconstructions ($4$ photons/px), Monte-Carlo checks of the estimator against its CRB. Saves `data/WZL_Fig4_syn.npz`.

In [ ]:
spec = OBJECTS["syn2"]; L = spec["L"]
TOTAL_PHOTONS = spec["nph"] * N**2
t0 = time.time()
print(f"=== syn2 ({spec['label']})  nph={spec['nph']}  optics tol={SYN_OPTICS_TOL} ===", flush=True)

A_states, B, K, psi, Nd = optics_exact_2d(L, tol=SYN_OPTICS_TOL)
bg   = make_object("blank")                      # flat background = the blank object
trn  = np.trace(object_operator(bg, A_states))
rho0 = object_operator(bg, A_states) / trn

obj      = make_object("syn2")
rho_full = object_operator(obj, A_states) / trn
C_gt     = dctn(obj, type=2, norm="ortho")

modes, drho = select_modes(obj, spec["nmodes"], L, trn, A_states)
M = len(modes)
print(f"  K={K}  M={M}  SDP block = {(M+1)*K}\n  modes={modes}", flush=True)

donn_val, V, masks = train_donn(B, Nd, rho0, drho)

qcrb = qcrb_scalar(rho0, drho)
nh, nh_K = nhcrb_sdp(rho0, drho) if spec["nhcrb"] else (None, None)

# DONN arm
u0MP, dMP, FMP = donn_calibration(V, rho0, drho)
diag_FMPi = np.diag(np.linalg.inv(FMP))
Wm = np.linalg.solve(FMP, dMP / u0MP)
u_full = np.clip(np.real(np.einsum("ip,ij,jp->p", V, rho_full, V.conj())), 0, None)
counts = np.random.default_rng(7).poisson(TOTAL_PHOTONS * u_full)
a_hat  = np.linalg.solve(FMP, (dMP / u0MP) @ (counts / TOTAL_PHOTONS - u0MP))
C_donn = np.zeros_like(C_gt); C_donn[0, 0] = C_gt[0, 0]
for (ux, uy), amp in zip(modes, a_hat):
    C_donn[uy, ux] = amp_to_dct(amp, ux, uy, L)
rec_donn = idctn(C_donn, type=2, norm="ortho")

# DI arm: same matched (locally-unbiased linear) estimator, no DONN
u0_di, dM_di, F_di, nb = di_machinery(modes, L, psi)
diag_Fdii = np.diag(np.linalg.inv(F_di))
Wm_di = np.linalg.solve(F_di, dM_di / u0_di)
di_val = np.trace(np.linalg.inv(F_di))
u_full_di = (obj.ravel() @ (psi**2)) / nb
counts_di = np.random.default_rng(11).poisson(TOTAL_PHOTONS * u_full_di)
a_hat_di  = Wm_di @ (counts_di / TOTAL_PHOTONS - u0_di)
C_di = np.zeros_like(C_gt); C_di[0, 0] = C_gt[0, 0]
for (ux, uy), amp in zip(modes, a_hat_di):
    C_di[uy, ux] = amp_to_dct(amp, ux, uy, L)
rec_di = idctn(C_di, type=2, norm="ortho")

# Monte Carlo check of the matched estimator, both arms
est = np.stack([Wm @ (np.random.default_rng(100 + t).poisson(TOTAL_PHOTONS * u_full)
                      / TOTAL_PHOTONS - u0MP) for t in range(MC_RUNS)])
mc_ratio = float(np.mean(est.std(axis=0) / np.sqrt(diag_FMPi / TOTAL_PHOTONS)))
est_di = np.stack([Wm_di @ (np.random.default_rng(500_000 + t).poisson(TOTAL_PHOTONS * u_full_di)
                      / TOTAL_PHOTONS - u0_di) for t in range(MC_RUNS)])
mc_ratio_di = float(np.mean(est_di.std(axis=0) / np.sqrt(diag_Fdii / TOTAL_PHOTONS)))

err_donn = np.linalg.norm(rec_donn - obj) / np.linalg.norm(obj)
err_di   = np.linalg.norm(rec_di   - obj) / np.linalg.norm(obj)

np.savez(SYN_NPZ,
         obj=obj, C_gt=C_gt, modes=np.array(modes), rec_donn=rec_donn, rec_di=rec_di,
         FMP=FMP, F_di=F_di, qcrb=qcrb, nhcrb=(np.nan if nh is None else nh),
         donn=donn_val, di=di_val, err_donn=err_donn, err_di=err_di,
         mc_ratio=mc_ratio, mc_ratio_di=mc_ratio_di, K=K, L=L,
         masks=np.stack([m.detach().cpu().numpy() for m in masks]))

print(f"  QCRB={qcrb:.1f}  NHCRB={'-' if nh is None else f'{nh:.1f}'}  "
      f"DONN={donn_val:.1f}  DI={di_val:.1f}  (DI/DONN={di_val/donn_val:.3f}x)")
print(f"  rel err: di={err_di:.4f}  donn={err_donn:.4f}   "
      f"MC/CRB: donn={mc_ratio:.3f} di={mc_ratio_di:.3f}   [{time.time()-t0:.0f}s]", flush=True)
print(f"  saved {SYN_NPZ}", flush=True)

### Shared device (atom array + diatom) — **[slow]**

Trained on the **blank** object (flat background $a_0$), so everything here is object-independent: full-band mode set, one DONN, calibrations for both channels, QCRB/DONN/DI bounds. Saves `data/WZL_Fig4_device.npz` (masks, `V`, calibrations) — a heavy local cache used only by the next cell; the figure never reads it.

In [ ]:
# shared device for the atom-array and diatom rows.  Run ONCE.
# The DONN is trained on the BLANK object, so one set of masks, calibrations and bounds serves all objects on this FOV.

_Ls = {spec["L"] for spec in OBJECTS.values()}
assert len(_Ls) == 1, f"all objects must share one FOV; got L values {_Ls}"
FOV_L = _Ls.pop()

t0 = time.time()
print(f"=== device: trained on 'blank'  L={FOV_L}  optics tol={DEVICE_OPTICS_TOL} ===", flush=True)

A_states, B, K, psi, Nd = optics_exact_2d(FOV_L, tol=DEVICE_OPTICS_TOL)
bg   = make_object("blank")                      # the blank object defines the device
trn  = np.trace(object_operator(bg, A_states))
rho0 = object_operator(bg, A_states) / trn

modes, drho = select_all_modes(FOV_L, NA, lam, N, trn, A_states)
M = len(modes)
print(f"  K={K}  M={M}", flush=True)

donn_val, V, masks = train_donn(B, Nd, rho0, drho)

qcrb = qcrb_scalar(rho0, drho)

u0MP, dMP, FMP = donn_calibration(V, rho0, drho)
u0_di, dM_di, F_di, nb = di_machinery(modes, FOV_L, psi)
di_val = np.trace(np.linalg.inv(F_di))

print(f"  QCRB={qcrb:.1f}  DONN={donn_val:.1f}  DI={di_val:.1f}  "
      f"(DI/DONN={di_val/donn_val:.3f}x)   [{time.time()-t0:.0f}s]", flush=True)

# one self-contained device file (everything the objects cell needs)
np.savez(DEVICE_NPZ,
         masks=np.stack([m.detach().cpu().numpy() for m in masks]), V=V,
         modes=np.array(modes), u0MP=u0MP, dMP=dMP, FMP=FMP,
         u0_di=u0_di, dM_di=dM_di, F_di=F_di, nb=nb,
         qcrb=qcrb, donn=donn_val, di=di_val, K=K, Nd=Nd, trn=trn, L=FOV_L)
print(f"  saved {DEVICE_NPZ}", flush=True)

### Atom array and diatom rows

Re-run freely after editing `OBJECTS` / `nph` — reuses the saved device masks, never retrains. Poisson counts, matched-estimator reconstructions for both channels, Monte-Carlo checks. Saves `data/WZL_Fig4_objects.npz`.

In [ ]:
# rows 2-3 (atom array, diatom).  Re-run freely after editing OBJECTS / nph.
# Uses the masks from the device cell; never retrains.
_Ls = {spec["L"] for spec in OBJECTS.values()}
assert len(_Ls) == 1, f"all objects must share one FOV; got L values {_Ls}"
FOV_L = _Ls.pop(); L = FOV_L

# optics are deterministic in L -- rebuilt (cached within a session), not stored
A_states, B, _K, psi, _Nd = optics_exact_2d(FOV_L, tol=DEVICE_OPTICS_TOL)

d = np.load(DEVICE_NPZ)
K, Nd, trn = int(d["K"]), int(d["Nd"]), float(d["trn"])
assert (K, Nd) == (_K, _Nd), f"device file (K={K},Nd={Nd}) != rebuilt optics (K={_K},Nd={_Nd})"
modes  = [tuple(int(v) for v in m) for m in d["modes"]]
V      = d["V"]
u0MP, dMP, FMP     = d["u0MP"], d["dMP"], d["FMP"]
u0_di, dM_di, F_di = d["u0_di"], d["dM_di"], d["F_di"]
nb = float(d["nb"])
qcrb, donn_val, di_val = float(d["qcrb"]), float(d["donn"]), float(d["di"])
diag_FMPi = np.diag(np.linalg.inv(FMP))
diag_Fdii = np.diag(np.linalg.inv(F_di))
Wm    = np.linalg.solve(FMP, dMP / u0MP)
Wm_di = np.linalg.solve(F_di, dM_di / u0_di)
print(f"loaded {DEVICE_NPZ}   K={K} Nd={Nd} M={len(modes)}", flush=True)

results = {}
for name in ROW_OBJECTS:
    spec = OBJECTS[name]
    if name == "syn2":
        continue                      # the cell-like row has its own pipeline above
    t0 = time.time()
    TOTAL_PHOTONS = spec["nph"] * N**2
    print(f"\n=== {name} ({spec['label']})  nph={spec['nph']} ===", flush=True)

    obj      = make_object(name)
    rho_full = object_operator(obj, A_states) / trn
    C_gt     = dctn(obj, type=2, norm="ortho")

    # DONN arm
    u_full = np.clip(np.real(np.einsum("ip,ij,jp->p", V, rho_full, V.conj())), 0, None)
    counts = np.random.default_rng(7).poisson(TOTAL_PHOTONS * u_full)
    a_hat  = np.linalg.solve(FMP, (dMP / u0MP) @ (counts / TOTAL_PHOTONS - u0MP))
    C_donn = np.zeros_like(C_gt); C_donn[0, 0] = C_gt[0, 0]
    for (ux, uy), amp in zip(modes, a_hat):
        C_donn[uy, ux] = amp_to_dct(amp, ux, uy, L)
    rec_donn = idctn(C_donn, type=2, norm="ortho")

    # DI arm: same matched estimator
    u_full_di = (obj.ravel() @ (psi**2)) / nb
    counts_di = np.random.default_rng(11).poisson(TOTAL_PHOTONS * u_full_di)
    a_hat_di  = Wm_di @ (counts_di / TOTAL_PHOTONS - u0_di)
    C_di = np.zeros_like(C_gt); C_di[0, 0] = C_gt[0, 0]
    for (ux, uy), amp in zip(modes, a_hat_di):
        C_di[uy, ux] = amp_to_dct(amp, ux, uy, L)
    rec_di = idctn(C_di, type=2, norm="ortho")

    # Monte Carlo check of the matched estimator, both arms
    est = np.stack([Wm @ (np.random.default_rng(100 + t).poisson(TOTAL_PHOTONS * u_full)
                          / TOTAL_PHOTONS - u0MP) for t in range(MC_RUNS)])
    mc_ratio = float(np.mean(est.std(axis=0) / np.sqrt(diag_FMPi / TOTAL_PHOTONS)))
    est_di = np.stack([Wm_di @ (np.random.default_rng(500_000 + t).poisson(TOTAL_PHOTONS * u_full_di)
                          / TOTAL_PHOTONS - u0_di) for t in range(MC_RUNS)])
    mc_ratio_di = float(np.mean(est_di.std(axis=0) / np.sqrt(diag_Fdii / TOTAL_PHOTONS)))

    results[name] = dict(
        obj=obj, C_gt=C_gt, rec_donn=rec_donn, rec_di=rec_di,
        qcrb=qcrb, donn=donn_val, di=di_val, L=L,
        err_donn=np.linalg.norm(rec_donn - obj) / np.linalg.norm(obj),
        err_di=np.linalg.norm(rec_di - obj) / np.linalg.norm(obj),
        mc_ratio=mc_ratio, mc_ratio_di=mc_ratio_di, K=K)

    r = results[name]
    print(f"  rel err: di={r['err_di']:.4f}  donn={r['err_donn']:.4f}   "
          f"MC/CRB: donn={mc_ratio:.3f} di={mc_ratio_di:.3f}   [{time.time()-t0:.0f}s]", flush=True)

# device_* fields: the small shared Fisher data the figure cell needs, so the
# heavy device cache (V, dMP, dM_di: >100 MB at full size) is never read again
np.savez(OBJECTS_NPZ,
         device_FMP=FMP, device_F_di=F_di, device_modes=np.array(modes),
         **{f"{n}_{k}": v for n, dd in results.items() for k, v in dd.items() if v is not None})
print(f"\nsaved {OBJECTS_NPZ}")

### Figure — **[fast]**

Reads only `data/WZL_Fig4_syn.npz` and `data/WZL_Fig4_objects.npz`.

In [ ]:
# reads only WZL_Fig4_syn.npz + WZL_Fig4_objects.npz
syn = dict(np.load(SYN_NPZ))
npz = np.load(OBJECTS_NPZ, allow_pickle=True)

results = {"syn2": {k: v for k, v in syn.items()}}
for key in npz.files:
    if key.startswith("device_"):
        continue
    name, field = key.split("_", 1)
    results.setdefault(name, {})[field] = npz[key]
for name, r in results.items():
    for k, v in list(r.items()):
        if isinstance(v, np.ndarray) and v.shape == ():
            r[k] = v.item()
results["syn2"]["nhcrb"] = None if np.isnan(results["syn2"]["nhcrb"]) else float(results["syn2"]["nhcrb"])

# CRB-ratio inputs
ratio_src = {"syn2": (syn["FMP"], syn["F_di"], [tuple(int(v) for v in m) for m in syn["modes"]])}
_dev_modes = [tuple(int(v) for v in m) for m in npz["device_modes"]]
for name in ("atoms", "diatom"):
    ratio_src[name] = (npz["device_FMP"], npz["device_F_di"], _dev_modes)

rows = [(n, OBJECTS[n]["label"], OBJECTS[n]["show_err"]) for n in ROW_OBJECTS]

TITLE_FS = 17
LABEL_FS = 15
TICK_FS  = 14
ANNOT_FS = 13
CELL_FS  = 6

def _blank_zero(v, pos):
    return "" if np.isclose(v, 0) else f"{v:.2f}"

def set_fov_ticks(a, L):
    a.set_xticks([0, L / 3, 2 * L / 3, L])
    a.set_yticks([0, L / 3, 2 * L / 3, L])
    a.xaxis.set_major_formatter(FuncFormatter(_blank_zero))
    a.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))

fig, ax = plt.subplots(
    len(rows), 5, figsize=(22, 4.4 * len(rows)),
    gridspec_kw=dict(wspace=0.30, hspace=0.28),
    squeeze=False,          # <- always 2-D, any row count
)

for i, (name, rowlab, show_err) in enumerate(rows):
    r = results[name]
    L = r["L"]
    m_cut = 2 * NA / lam * 2 * L

    obj = r["obj"]
    vmin, vmax = obj.min(), obj.max()
    ext = [0, L, 0, L]

    # (i) Ground truth
    a = ax[i, 0]
    a.imshow(obj, cmap="inferno", origin="lower", vmin=vmin, vmax=vmax, extent=ext)
    a.set_box_aspect(1)
    set_fov_ticks(a, L)
    a.tick_params(labelsize=TICK_FS)
    a.set_ylabel(rowlab + "\n\n$y$ ($\\mu$m)", fontsize=LABEL_FS)
    a.set_xlabel("$x$ ($\\mu$m)", fontsize=LABEL_FS)
    if i == 0:
        a.set_title("(i) Ground truth", fontsize=TITLE_FS, pad=16)

    # (ii) Per-mode CRB variance ratio  DONN / DI
    a = ax[i, 1]
    a.set_box_aspect(1)
    FMP_i, Fdi_i, modes_i = ratio_src[name]
    assert len(modes_i) == FMP_i.shape[0] == Fdi_i.shape[0], \
        "modes / Fisher size mismatch -- npz is stale, re-run the pipeline cells"

    vD = np.diag(np.linalg.inv(FMP_i))
    vI = np.diag(np.linalg.inv(Fdi_i))
    if vD.min() <= 0 or vI.min() <= 0:
        raise ValueError("non-positive CRB variance: F is not positive definite")
    ratio = vD / vI

    mmax = max(max(m) for m in modes_i) + 2
    R = np.full((mmax, mmax), np.nan)
    for (ux, uy), rr in zip(modes_i, ratio):
        if ux < mmax and uy < mmax:
            R[uy, ux] = rr

    rmax = float(max(ratio.max(), 1.0 / ratio.min()))
    rmax = min(max(rmax, 1.2), 100.0)             # keep the colour scale readable
    cmap = plt.cm.RdBu_r.copy(); cmap.set_bad("0.9")

    im = a.imshow(R, cmap=cmap, origin="lower",
                  norm=LogNorm(vmin=1.0 / rmax, vmax=rmax),
                  extent=[-.5, mmax - .5, -.5, mmax - .5])
    th = np.linspace(0, np.pi / 2, 200)
    a.plot(m_cut * np.cos(th), m_cut * np.sin(th), "k-", lw=1.6)
    a.plot(m_cut / 2 * np.cos(th), m_cut / 2 * np.sin(th), "k--", lw=1.3)
    if mmax <= 14:                                # annotate only while legible
        for (ux, uy), rr in zip(modes_i, ratio):
            if ux < mmax and uy < mmax:
                a.text(ux, uy, f"{rr:.2f}", ha="center", va="center", fontsize=CELL_FS,
                       color="w" if (rr > rmax**0.5 or rr < rmax**-0.5) else "k")
    a.set_xlim(-.5, mmax - .5)
    a.set_ylim(-.5, mmax - .5)
    a.set_ylabel("mode index $m_y$", fontsize=LABEL_FS)
    if i == 0:
        a.set_title("(ii) CRB ratio DONN/DI", fontsize=TITLE_FS, pad=16)

    # (iii) DI reconstruction / (iv) DONN reconstruction
    for col, (img, err, title) in enumerate(
        [(r["rec_di"], r["err_di"], "(iii) DI reconstruction"),
         (r["rec_donn"], r["err_donn"], "(iv) DONN reconstruction")],
        start=2
    ):
        a = ax[i, col]
        a.imshow(img, cmap="inferno", origin="lower", vmin=vmin, vmax=vmax, extent=ext)
        a.set_box_aspect(1)
        set_fov_ticks(a, L)
        a.tick_params(labelsize=TICK_FS)
        if show_err:
            a.text(
                0.04, 0.94, f"rel. err. {err:.3f}",
                transform=a.transAxes, color="w", fontsize=ANNOT_FS, va="top",
                bbox=dict(fc="k", alpha=0.55, pad=2, ec="none")
            )
        if i == 0:
            a.set_title(title, fontsize=TITLE_FS, pad=16)

    # (v) Precision bounds
    a = ax[i, 4]
    a.set_box_aspect(1)
    if r.get("nhcrb", None) is not None:
        labels = ["QCRB", "NHCRB", "DONN", "DI"]
        vals   = [r["qcrb"], r["nhcrb"], r["donn"], r["di"]]
        cols   = ["crimson", "C0", "darkorange", "0.45"]
    else:
        labels = ["QCRB", "DONN", "DI"]
        vals   = [r["qcrb"], r["donn"], r["di"]]
        cols   = ["crimson", "darkorange", "0.45"]

    a.bar(np.arange(len(vals)), vals, color=cols, width=0.62)
    a.set_yscale("log")
    a.set_ylim(min(vals) / 2.2, max(vals) * 7)
    a.set_xticks(np.arange(len(vals)))
    a.set_xticklabels(labels, fontsize=TICK_FS)
    a.tick_params(axis="y", labelsize=TICK_FS)
    a.set_ylabel(r"Per-photon Tr$(F^{-1})$", fontsize=LABEL_FS, labelpad=1)
    a.grid(True, axis="y", which="both", ls="-.", alpha=0.3)
    if i == 0:
        a.set_title("(v) Precision bounds", fontsize=TITLE_FS, pad=16)

    ax[i, 1].set_xlabel("mode index $m_x$" if i == 0 else "", fontsize=LABEL_FS)
    ax[i, 1].tick_params(labelsize=TICK_FS)

fig.savefig("figures/WZL_Fig4.png", dpi=300, bbox_inches="tight")
fig.savefig("figures/WZL_Fig4.svg", bbox_inches="tight")
print("saved figures/WZL_Fig4.png / .svg")

### DI camera reference (Fig 4 in main draft is then rendered using CorelDraw)

In [ ]:
# DI camera reference
CROP_TO_FOV = True
FLATFIELD   = True      # divide by the flat-background frame
SAVE_NPZ    = True

FOV_L = {spec["L"] for spec in OBJECTS.values()}.pop(); L = FOV_L
dx  = L / N
*_, psi, Nd = optics_exact_2d(L, tol=DEVICE_OPTICS_TOL)   # cached within the session
off = (Nd - N) // 2                        # xd[off:off+N] == x exactly
ext = [0, L, 0, L] if CROP_TO_FOV else [-off*dx, (Nd-off)*dx]*2

P  = psi**2                                                  # ~0.5 GB at N=64, Nd=128
ub = make_object("blank").ravel() @ P                        # blank-object frame, object units
nb = ub.sum()
u0 = (ub / nb).reshape(Nd, Nd)

cmap = plt.cm.inferno.copy(); cmap.set_bad("k")
di_cams, rows = {}, [(n, OBJECTS[n]) for n in ROW_OBJECTS]
fig, ax = plt.subplots(len(rows), 2, figsize=(9, 4.4*len(rows)), dpi=150,
                       gridspec_kw=dict(wspace=0.25, hspace=0.28), squeeze=False)

for i, (name, spec) in enumerate(rows):
    obj = make_object(name)
    TOTAL_PHOTONS = spec["nph"] * N**2
    u      = (obj.ravel() @ P) / nb                           # identical to the DI arm
    counts = np.random.default_rng(11).poisson(TOTAL_PHOTONS * u).reshape(Nd, Nd)

    if FLATFIELD:
        cam = counts / (TOTAL_PHOTONS * np.maximum(u0, 1e-12)) * (a0 / L**2)
        cam[u0 < 0.02 * u0.max()] = np.nan                    # corners: nothing to divide by
    else:
        cam = counts * (nb / TOTAL_PHOTONS)

    di_cams[name] = dict(di_cam=cam[off:off+N, off:off+N], di_cam_full=cam)
    view = cam[off:off+N, off:off+N] if CROP_TO_FOV else cam

    vmin, vmax = obj.min(), obj.max()
    for a, img, ttl in [(ax[i, 0], obj, "Ground truth"), (ax[i, 1], view, "DI camera")]:
        a.imshow(img, cmap=cmap, origin="lower", vmin=vmin, vmax=vmax, extent=ext)
        a.set_box_aspect(1)
        a.set_xticks([])
        a.set_yticks([])
    ax[i, 0].set_ylabel(spec["label"], fontsize=13)
    if i == 0:
        ax[i, 0].set_title("Ground truth", fontsize=15, pad=12)
        ax[i, 1].set_title("DI camera (flat-fielded)", fontsize=15, pad=12)

    cc = counts[off:off+N, off:off+N]
    print(f"{name}: nph={spec['nph']}  counts/px={cc.mean():.0f}  "
          f"cam [{np.nanmin(view):.3f}, {np.nanmax(view):.3f}]  window [{vmin:.3f}, {vmax:.3f}]  "
          f"peak at {(np.nanmax(view)-vmin)/(vmax-vmin):.2f} of range", flush=True)

del P
plt.tight_layout(); plt.show()

if SAVE_NPZ:
    np.savez(DICAM_NPZ,
             **{f"{n}_{k}": v for n, dd in di_cams.items() for k, v in dd.items()})
    print(f"saved {DICAM_NPZ}")